# Tracker de clasificación — LaLiga (bumpy chart estilo mplsoccer)

Descarga jornada a jornada los resultados de LaLiga desde la API pública de Sofascore, calcula la clasificación acumulada y genera un **bumpy chart** (gráfico de evolución de posiciones) con el mismo estilo que el ejemplo de la galería de `mplsoccer`.

**Para adaptarlo a la temporada 26/27 (o cualquier otra) solo hay que cambiar una variable**: `SEASON_LABEL` en la celda de configuración. El resto del notebook no requiere ningún cambio.

> Nota: los endpoints de Sofascore usados aquí no son oficiales (no hay API pública documentada por Sofascore). Pueden cambiar sin previo aviso; si algo deja de funcionar, revisa primero si la estructura del JSON de respuesta ha cambiado.

In [1]:
# Dependencias necesarias (descomenta la línea si es la primera vez que ejecutas el notebook)
# Requiere tener Google Chrome instalado en el equipo (undetected-chromedriver lo pilota).
!pip install undetected-chromedriver selenium pandas numpy mplsoccer highlight_text --quiet

In [2]:
!pip install undetected-chromedriver selenium

In [3]:
import os
import json
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import undetected_chromedriver as uc
import matplotlib.pyplot as plt
from mplsoccer import Bumpy
from highlight_text import fig_text

## 1. Configuración

Único bloque que hay que tocar entre temporadas. `TOURNAMENT_ID = 8` corresponde a LaLiga en Sofascore (`sofascore.com/football/tournament/spain/laliga/8`); si en el futuro quieres reutilizar el notebook para otra liga, solo hay que cambiar este ID.

In [4]:
TOURNAMENT_ID = 8              # LaLiga en Sofascore
SEASON_LABEL  = "25/26"        # <-- Cambiar a "26/27" cuando arranque la nueva temporada
MAX_ROUNDS    = 38             # Jornadas totales de LaLiga (20 equipos)
REQUEST_DELAY = 1.0            # Segundos entre peticiones, para no saturar la API
CACHE_DIR     = "data"         # Carpeta donde se cachean las jornadas ya descargadas
HEADLESS      = False          # Poner a True cuando confirmes que funciona sin ventana visible

BASE_URL = "https://api.sofascore.com/api/v1"

os.makedirs(CACHE_DIR, exist_ok=True)

## 2. Navegador Chrome real (curl_cffi no es suficiente)

Sofascore ha reforzado su protección: además de comprobar la huella TLS (lo que resolvía `curl_cffi`), ahora también puede lanzar un **reto de verificación en JavaScript** (Cloudflare Managed Challenge / Turnstile) que solo un navegador real puede resolver, porque necesita ejecutar el JavaScript del reto y simular una sesión de navegación genuina.

Por eso se usa [`undetected-chromedriver`](https://github.com/ultrafunkamsterdam/undetected-chromedriver): pilota un **Chrome real instalado en tu equipo** (con los parches necesarios para no ser detectado como automatizado) y navega directamente a cada URL de la API, dejando que el propio Chrome resuelva el reto si aparece. Necesitas tener **Google Chrome instalado** — no hace falta descargar nada más, `undetected-chromedriver` gestiona el driver.

Se deja `HEADLESS = False` por defecto (verás la ventana de Chrome abrirse y navegar sola): los navegadores headless tienen más papeletas de ser detectados por este tipo de retos. Una vez confirmes que funciona, puedes probar `HEADLESS = True` para que no abra ventana.

In [5]:
def build_driver(headless: bool = HEADLESS) -> uc.Chrome:
    options = uc.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    return uc.Chrome(options=options)


CF_MARKERS = ("just a moment", "attention required", "checking your browser",
              "verifying you are human")


def fetch_json(driver: uc.Chrome, url: str, max_retries: int = 4, wait_seconds: float = 4.0) -> dict:
    """
    Navega con el Chrome real a una URL de la API y devuelve el JSON parseado.
    Si Cloudflare muestra una pantalla de verificación, espera y reintenta.
    """
    for intento in range(max_retries):
        driver.get(url)
        time.sleep(wait_seconds)

        title = (driver.title or "").lower()
        if any(marker in title for marker in CF_MARKERS):
            time.sleep(wait_seconds * (intento + 1))
            continue

        body_text = driver.execute_script("return document.body.innerText")
        try:
            return json.loads(body_text)
        except json.JSONDecodeError:
            time.sleep(wait_seconds)
            continue

    raise RuntimeError(f"No se pudo obtener un JSON válido de {url} tras {max_retries} intentos")


driver = build_driver()

SessionNotCreatedException: Message: session not created: cannot connect to chrome at 127.0.0.1:50679
from session not created: This version of ChromeDriver only supports Chrome version 151
Current browser version is 150.0.7871.187; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
	undetected_chromedriver!GetHandleVerifier [0xabd993+fca3]
	undetected_chromedriver!GetHandleVerifier [0xabd9d4+fce4]
	undetected_chromedriver!(No symbol) [0x899240]
	undetected_chromedriver!(No symbol) [0x8d5c12]
	undetected_chromedriver!(No symbol) [0x8d4c3c]
	undetected_chromedriver!(No symbol) [0x8cae85]
	undetected_chromedriver!(No symbol) [0x8caca6]
	undetected_chromedriver!(No symbol) [0x91190f]
	undetected_chromedriver!(No symbol) [0x911127]
	undetected_chromedriver!(No symbol) [0x905626]
	undetected_chromedriver!(No symbol) [0x8d87f9]
	undetected_chromedriver!(No symbol) [0x8d95c4]
	undetected

## 3. Localizar el `season_id` correcto

Sofascore identifica cada temporada con un `season_id` interno (distinto cada año). **Se busca explícitamente la temporada que coincide con `SEASON_LABEL`** en lugar de tomar la primera de la lista: en pretemporada, Sofascore ya crea el placeholder de la temporada siguiente (p. ej. "26/27") sin partidos jugados, y tomar `seasons[0]` a ciegas apuntaría ahí en vez de a la temporada activa.

In [ ]:
def get_season_id(tournament_id: int, season_label: str, driver: uc.Chrome) -> int:
    """
    Devuelve el season_id de Sofascore cuyo campo 'year' coincide con season_label (ej. '25/26').
    """
    url = f"{BASE_URL}/unique-tournament/{tournament_id}/seasons"
    data = fetch_json(driver, url)
    seasons = data["seasons"]

    for s in seasons:
        if s["year"].replace(" ", "") == season_label.replace(" ", ""):
            return s["id"]

    disponibles = ", ".join(s["year"] for s in seasons[:8])
    raise ValueError(f"No se encontró la temporada '{season_label}'. Disponibles: {disponibles}")


SEASON_ID = get_season_id(TOURNAMENT_ID, SEASON_LABEL, driver)
print(f"Temporada '{SEASON_LABEL}' -> season_id = {SEASON_ID}")

NameError: name 'driver' is not defined

## 4. Descargar resultados jornada a jornada (con caché)

Se descarga cada jornada por separado, reutilizando la misma ventana de Chrome, y se guarda en `data/` para no tener que volver a pedirlas si se re-ejecuta el notebook. La descarga se detiene sola en la primera jornada aún no disputada, y al terminar se cierra el navegador.

In [ ]:
def get_round_events(tournament_id: int, season_id: int, round_number: int,
                      driver: uc.Chrome):
    """
    Descarga los partidos de una jornada. Devuelve None si la jornada todavía no existe
    (permite parar la descarga automáticamente al llegar al presente).
    """
    url = f"{BASE_URL}/unique-tournament/{tournament_id}/season/{season_id}/events/round/{round_number}"
    data = fetch_json(driver, url)
    if "error" in data:
        return None
    return data.get("events", [])

In [ ]:
cache_file = os.path.join(CACHE_DIR, f"laliga_{SEASON_LABEL.replace('/', '_')}_rounds.json")

if os.path.exists(cache_file):
    with open(cache_file, "r", encoding="utf-8") as f:
        rounds_data = {int(k): v for k, v in json.load(f).items()}
    print(f"Cargadas {len(rounds_data)} jornadas desde caché ({cache_file})")
else:
    rounds_data = {}

try:
    for round_number in range(1, MAX_ROUNDS + 1):
        if round_number in rounds_data:
            continue
        events = get_round_events(TOURNAMENT_ID, SEASON_ID, round_number, driver)
        if events is None:
            print(f"Jornada {round_number}: aún no disponible en Sofascore, se detiene la descarga.")
            break
        rounds_data[round_number] = events
        finished = sum(1 for e in events if e.get("status", {}).get("type") == "finished")
        print(f"Jornada {round_number}: {finished}/{len(events)} partidos finalizados")
        time.sleep(REQUEST_DELAY)
finally:
    driver.quit()  # ya no hace falta el navegador, liberamos el proceso de Chrome

with open(cache_file, "w", encoding="utf-8") as f:
    json.dump(rounds_data, f, ensure_ascii=False)
print(f"\nGuardado en caché: {cache_file}")

## 5. Calcular la clasificación acumulada jornada a jornada

A partir de los resultados se reconstruye la tabla tras cada jornada. **Desempate aplicado: puntos → diferencia de goles → goles a favor.** Es la aproximación estándar para este tipo de gráficos; no reproduce el desempate oficial de LaLiga por enfrentamiento directo, así que en empates triples a puntos/diferencia/goles la posición mostrada puede variar en un puesto respecto a la tabla oficial.

In [ ]:
def build_standings_progression(rounds_data: dict) -> dict:
    """
    Devuelve {nombre_equipo: [posicion_jornada_1, posicion_jornada_2, ...]}
    """
    stats = defaultdict(lambda: {"pts": 0, "gf": 0, "ga": 0})
    progression = defaultdict(list)
    all_teams = set()

    for round_number in sorted(rounds_data.keys()):
        for match in rounds_data[round_number]:
            if match.get("status", {}).get("type") != "finished":
                continue
            home, away = match["homeTeam"]["name"], match["awayTeam"]["name"]
            hs = match["homeScore"]["current"]
            as_ = match["awayScore"]["current"]
            all_teams.update([home, away])

            stats[home]["gf"] += hs; stats[home]["ga"] += as_
            stats[away]["gf"] += as_; stats[away]["ga"] += hs

            if hs > as_:
                stats[home]["pts"] += 3
            elif hs < as_:
                stats[away]["pts"] += 3
            else:
                stats[home]["pts"] += 1; stats[away]["pts"] += 1

        table = sorted(
            all_teams,
            key=lambda t: (-stats[t]["pts"], -(stats[t]["gf"] - stats[t]["ga"]), -stats[t]["gf"])
        )
        for position, team in enumerate(table, start=1):
            progression[team].append(position)

    return dict(progression)


standings_progression = build_standings_progression(rounds_data)
print(f"Equipos: {len(standings_progression)} | Jornadas calculadas: "
      f"{len(next(iter(standings_progression.values()), []))}")

In [ ]:
# Vista rápida en tabla (una fila por equipo, una columna por jornada)
df_progression = pd.DataFrame(standings_progression).T
df_progression.columns = [f"J{c}" for c in df_progression.columns]
df_progression.sort_values(df_progression.columns[-1])

## 6. Colores de equipos y selección de destacados

`TEAM_COLORS` recoge el color principal de los equipos habituales de LaLiga (ajusta o añade los que falten según los equipos de cada temporada — sube/bajan equipos cada año). `EQUIPOS_DESTACADOS` es el subconjunto que se resalta en el gráfico; el resto de equipos queda en gris de fondo, igual que en el ejemplo de referencia.

In [ ]:
TEAM_COLORS = {
    "Real Madrid": "#FEBE10",
    "Barcelona": "#A50044",
    "Atletico Madrid": "#CB3524",
    "Athletic Club": "#EE2523",
    "Real Sociedad": "#0068A8",
    "Real Betis": "#00954C",
    "Sevilla": "#D8022A",
    "Villarreal": "#FFE667",
    "Valencia": "#EE7A15",
    "Girona": "#CD2534",
    "Celta Vigo": "#8AC3EE",
    "Osasuna": "#D91A21",
    "Rayo Vallecano": "#EE2523",
    "Mallorca": "#CC1D30",
    "Getafe": "#005CA9",
    "Alaves": "#1F4396",
    "Las Palmas": "#FEE102",
    "Leganes": "#00518C",
    "Espanyol": "#0A63A8",
    "Valladolid": "#7B0044",
    "Elche": "#00954C",
    "Levante": "#B4131B",
}

# Equipos a resaltar en el bumpy chart -> edita esta lista a tu gusto
EQUIPOS_DESTACADOS = {
    team: color for team, color in TEAM_COLORS.items()
    if team in ("Real Madrid", "Barcelona", "Atletico Madrid")
}

## 7. Bumpy chart

Mismo estilo que el ejemplo de la galería de `mplsoccer` (fondo oscuro, líneas curvas, posiciones en el eje Y invertido, equipos destacados en color y el resto en gris).

In [ ]:
n_teams = len(standings_progression)
weeks_played = len(next(iter(standings_progression.values())))
match_day = [f"J{n}" for n in range(1, weeks_played + 1)]

bumpy = Bumpy(
    background_color="#1B1B1B",
    scatter_color="#282A2C",
    line_color="#3B3B3B",
    label_color="#F2F2F2",
    rotate_xticks=90,
    ticklabel_size=13,
    label_size=18,
    scatter_primary="o",
    show_right=True,
    plot_labels=True,
    alignment_yvalue=0.1,
    alignment_xvalue=0.06,
)

fig, ax = bumpy.plot(
    x_list=match_day,
    y_list=np.linspace(1, n_teams, n_teams).astype(int),
    values=standings_progression,
    secondary_alpha=0.35,
    highlight_dict=EQUIPOS_DESTACADOS,
    figsize=(20, 12),
    x_label="Jornada",
    y_label="Posición",
)

fig_text(
    x=0.14, y=0.97,
    s=f"<LaLiga {SEASON_LABEL}> — Evolución de la clasificación jornada a jornada",
    highlight_textprops=[{"color": "#F2F2F2", "weight": "bold"}],
    color="#F2F2F2", fontsize=22, fig=fig,
)

output_path = f"laliga_{SEASON_LABEL.replace('/', '_')}_bumpy_chart.png"
fig.savefig(output_path, dpi=150, facecolor=bumpy.background_color, bbox_inches="tight")
print(f"Gráfico guardado en: {output_path}")

## Notas para adaptar a la temporada 26/27

1. Cambia `SEASON_LABEL = "26/27"` en la celda de configuración y vuelve a ejecutar todo el notebook — el resto del código no necesita ningún cambio.
2. La caché en `data/` es por temporada (`laliga_26_27_rounds.json`), así que no pisa los datos de `25/26`. Cuanto más avance la temporada, más jornadas nuevas se irán añadiendo cada vez que reejecutes la celda de descarga.
3. Si sube o baja algún equipo respecto a la temporada anterior, añade su color a `TEAM_COLORS` (si no está, mplsoccer lo pintará igualmente, pero con el color gris por defecto en vez de uno propio).
4. Necesitas **Google Chrome instalado** en el equipo donde ejecutes el notebook — `undetected-chromedriver` lo pilota directamente. Si `HEADLESS = True` empieza a fallar, vuelve a `False` para ver qué está pasando en la ventana de Chrome.
5. Si Sofascore refuerza aún más su protección y esto deja de funcionar, la señal habitual es que `fetch_json` agote los reintentos con el título "Just a moment..." — en ese caso, sube `wait_seconds` en `fetch_json` para dar más margen a que se resuelva el reto.
6. Los endpoints usados (`/unique-tournament/.../seasons`, `.../events/round/...`) son internos y no oficiales: pueden cambiar de estructura sin aviso. Si una celda empieza a fallar con un `KeyError`, imprime el JSON crudo de la respuesta para ver si el nombre de algún campo ha cambiado.